# 1 · Read status without changing settings

Learn how to ask for software identity, operating state, power, temperature and fault evidence. This is the first lesson to run because all transmitted instructions are queries.

**Self-contained notebook · simulator default · optional human-operated hardware**

From the repository root, activate your virtual environment, install with `python -m pip install -e ".[tutorials]"`, then launch `python -m jupyterlab examples/tutorials`. Select this environment's Python kernel. Run cells top to bottom with Shift+Enter, or choose **Restart Kernel and Run All Cells**. See [setup and troubleshooting](README.md#start-here).

Every demonstration and hardware helper is defined below. The notebook imports only the controller library and Python's standard library; no tutorial script is loaded. Run cells from top to bottom.

[All four tutorials](README.md)

## 1. Command sequence

`?SV` → `status()` → `diagnostics()` → locally rejected write → communication close.

`status()` sends `?L, ?K, ?S, ?P, ?SP, ?D1C, ?D1T, ?D1HST, ?BT, ?LBOT, ?LBOSS, ?ET, ?VT, ?F`.
`diagnostics()` sends `?SV, ?HH, ?PSH, ?D1H, ?FH`.
The samples are sequential. A version string does not identify the model; V5 is caller configuration.

The controller supplies CR/LF framing and waits for each reply. Queries start with `?`; writes use `=`. No `OK` acknowledgment is assumed. Source: supplied [Verdi manual](../../verdi.manual_v5.pdf), Tables 5-1, 5-3 and 5-4; [protocol mapping](../../docs/PROTOCOL.md).

## 2. Import the controller API

Imports do not discover ports, open connections or send commands.

In [ ]:
"""Tutorial 1: read identity, status and diagnostics without changing settings."""

from coherent_verdi import (
    ControllerConfig,
    Model,
    Query,
    SimulatedTransport,
    Status,
    VerdiController,
    VerdiError,
    WritesDisabled,
)

## 3. Define the application steps

Each cell defines one small function. The complete sequence is run in section 4.

### 3.1 Read state and diagnostics

The queries below collect state and diagnostic values. Units stay visible in the printed output. Faults are reported without reset.

In [ ]:
def read_status(laser: VerdiController) -> Status:
    version = laser.query(Query.SOFTWARE)  # ?SV: query, not model discovery.
    status = laser.status()  # Fourteen sequential queries, not one atomic reading.
    diagnostics = laser.diagnostics()
    source = "SIMULATOR" if status.simulated else "HARDWARE"
    print(f"Source: {source}; configured model: {status.model}; software: {version}")
    print(
        f"State: {status.laser_state.name}; key ON: {status.keyswitch_on}; "
        f"shutter open: {status.shutter_open}"
    )
    print(f"Setpoint: {status.set_power_w:.4f} W; reported power: {status.power_w:.3f} W")
    print(f"LBO: {status.lbo_temp_c:.2f} degC / {status.lbo_servo.name}")
    print(f"Head operating hours: {diagnostics.head_hours:.1f} h")
    print(f"Active faults: {status.faults}; history: {diagnostics.fault_history}")
    if status.faults:
        print("STOP: active faults need diagnosis; this tutorial never resets or enables.")
    return status

## 4. Run the simulator

The fixture setup below is synthetic. The exercise target of 0.25 W and ceiling of 0.5 W are not physical safety limits.

### 4.1 Prepare the simulator demonstration

The function below owns the simulator and its controller lifetime. Each call starts a fresh exercise and leaves no worker or open session.

In [ ]:
def simulate_read_status() -> None:
    sim = SimulatedTransport(Model.V5)
    try:
        with VerdiController(sim, ControllerConfig(Model.V5)) as laser:
            read_status(laser)
            try:
                laser.set_power_w(0.25)
            except WritesDisabled:
                print("Expected WritesDisabled: the attempted write sent no command.")
    except VerdiError as exc:
        print(f"STOP: {type(exc).__name__}: {exc}. No retry; readings are incomplete.")
        raise
    print("Connection released; close() sends no laser commands.")

### 4.2 Execute the demonstration

Run this short cell to call the functions just defined. Rerun it to begin with a fresh simulator.

In [ ]:
simulate_read_status()

## 5. Expected simulator result

```text
Source: SIMULATOR; configured model: V5; software: SIMULATOR-0.1
State: STANDBY; key ON: False; shutter open: False
Setpoint: 0.0000 W; reported power: 0.000 W
LBO: 148.00 degC / LOCKED
Head operating hours: 100.0 h
Active faults: (); history: ()
Expected WritesDisabled: the attempted write sent no command.
```
The final line confirms communication was released. These temperatures and hours are fixture constants, not measurements.

## 6. Try one small change

Change `Model.V5` in **both** simulator and controller configuration to `Model.V2`, then rerun. The configured label changes; the synthetic temperatures remain the same. Do not turn on writes in this lesson.

## 7. Optional real-hardware session for a human operator

Complete the [hardware review procedure](../../HARDWARE_VALIDATION.md) first. Install `python -m pip install -e ".[tutorials,serial]"` in this environment. The hardware functions below are fully visible and call the application functions in section 3 directly.

Fill in the actual model, native port and matching baud. This lesson keeps writes disabled. Keep `RUN_HARDWARE=False` for ordinary Run All.

After you enable it, **CONNECT** permits the selected connection and one `?SV` query. Keep `IDENTIFY_ONLY=True` for the first check; set it to False only when broader reads are approved. **RUN** permits the displayed lesson. Any other answer cancels. These prompts confirm intent, not site approval.

### 7.1 Import serial interfaces

These imports alone perform no I/O. `contextmanager` lets a `with` block release the connection even on an exception.

In [ ]:
from collections.abc import Iterator
from contextlib import contextmanager

from coherent_verdi import SerialConfig, open_serial

### 7.2 Validate the power configuration

Validation happens before connection. Read-only sessions reject power settings; write sessions require an explicit approved ceiling and target.

In [ ]:
def hardware_power_config(
    model: Model,
    *,
    allow_writes: bool = False,
    target_w: float | None = None,
    power_limit_w: float | None = None,
    active_fault_clear_reply: str | None = None,
) -> ControllerConfig:
    """Validate explicit hardware power settings before opening a connection."""
    config = ControllerConfig(model, allow_writes, power_limit_w, active_fault_clear_reply)
    if allow_writes:
        if power_limit_w is None:
            raise ValueError("Supply the approved power_limit_w; there is no hardware default")
        if (
            isinstance(target_w, bool)
            or not isinstance(target_w, (int, float))
            or not 0 <= target_w <= config.effective_power_limit_w
            or float(f"{target_w:.4f}") > config.effective_power_limit_w
        ):
            raise ValueError("target_w must be finite, nonnegative and within the approved ceiling")
    elif target_w is not None or power_limit_w is not None:
        raise ValueError("Read-only lessons do not accept power settings")
    return config

### 7.3 Connect, identify and release

This helper prompts before opening, sends `?SV` first and closes communication when the `with` block ends. An error stops further commands. **Connection close is not physical shutdown**; use the operator's abort procedure when state is uncertain.

In [ ]:
@contextmanager
def hardware_connection(
    serial_config: SerialConfig,
    config: ControllerConfig,
) -> Iterator[VerdiController | None]:
    """Ask CONNECT, read ?SV first, and release the connection on leaving the block."""
    print(f"HARDWARE: {config.model}, port={serial_config.port}, baud={serial_config.baudrate}")
    print(f"Timeout: {serial_config.timeout_s} s. First interaction: ?SV only.")
    if input("With candidate/connection approval recorded, type CONNECT: ").strip() != "CONNECT":
        print("Cancelled before opening the port.")
        yield None
        return
    try:
        with VerdiController(open_serial(serial_config, hardware_allowed=True), config) as laser:
            print(f"Reported software: {laser.query(Query.SOFTWARE)}")
            yield laser
    except BaseException:
        print("STOP: session failed/interrupted. No reconnect, retries or blind cleanup writes.")
        print("Physical state may be unknown. Use the operator's approved abort procedure.")
        raise
    finally:
        print("Communication scope ended; releasing a connection does not shut down the laser.")

### 7.4 Enter the operator's settings

Nothing connects when you run this configuration cell. Set `RUN_HARDWARE=True` only for an attended, approved session. Restore False afterwards.

The manual does not specify the no-active-fault reply for `?F`. Keep `ACTIVE_FAULT_CLEAR_REPLY=None` until Stage 1 records its exact meaning for this firmware. Then enter the verified text here; otherwise a clear-looking response stops the physical session. `SYSTEM OK` is documented for `?FH` only. Simulator defaults use their explicit fixture convention.

In [ ]:
RUN_HARDWARE = False  # Change only for an approved, attended physical session.
HARDWARE_PORT = None  # Set to the operator-confirmed native port, e.g. "COM3".
HARDWARE_MODEL = None  # Set to Model.V2, Model.V5 or Model.V6 after checking the label.
HARDWARE_BAUDRATE = None  # Set to the actual front-panel baud rate.
HARDWARE_TIMEOUT_S = 1.0  # Adjust to the validated transaction deadline.
IDENTIFY_ONLY = True  # First approved interaction: ?SV only.
ACTIVE_FAULT_CLEAR_REPLY = None  # Set only after Stage 1 verifies the exact ?F clear text.

### 7.5 Run the visible hardware sequence

Identification-only mode stops after `?SV`. The full mode calls `read_status` from section 3.

The call completes in one cell; no connection waits between cells. A new run prompts again.

In [ ]:
if RUN_HARDWARE:
    serial_config = SerialConfig(
        HARDWARE_PORT, baudrate=HARDWARE_BAUDRATE, timeout_s=HARDWARE_TIMEOUT_S
    )
    config = hardware_power_config(
        HARDWARE_MODEL, active_fault_clear_reply=ACTIVE_FAULT_CLEAR_REPLY
    )
    print("Plan: read status and diagnostics; no writes.")
    with hardware_connection(serial_config, config) as laser:
        if laser is not None:
            if IDENTIFY_ONLY:
                print("Identification only: no further queries or writes.")
            elif input("Verify device/version and approved scope; type RUN: ").strip() == "RUN":
                read_status(laser)
            else:
                print("Cancelled after ?SV; no lesson commands sent.")
else:
    print("Hardware section skipped. Set explicit operator configuration to use it.")

## 8. What this establishes

Simulator runs verify software behavior, not physical response or calibration. A status sample is sequential, not an interlock or proof of a safe beam path. Review the [simulator assumptions](../../docs/SIMULATOR.md) and [operator guide](README.md#human-operated-hardware) before physical use.